# JAX Real Dataset Pipeline Notebook (With Offline Fallback)

This notebook adds a practical, dataset-centric workflow after your basics + advanced notebooks.

Goals:
- Load a **real dataset** when available
- Keep pipeline robust when offline
- Train a small model end-to-end in JAX/Flax
- Practice batching, normalization, augmentation, and metrics


## Section 1: Pipeline Strategy

Recommended split:
1. Data loading/parsing (CPU): dataset libs (`tensorflow_datasets`, HuggingFace, sklearn, local files)
2. Numeric batch transforms: JAX functions (`jit` where stable)
3. Model + optimization: Flax + Optax + JAX

In this notebook we try in order:
- TFDS MNIST
- sklearn digits
- synthetic fallback


In [ ]:
import jax
import jax.numpy as jnp
from jax import random, jit
import numpy as np
from flax import linen as nn
import optax

print('JAX:', jax.__version__)
print('Device:', jax.devices()[0])


## Section 2: Dataset Loader (MNIST -> Digits -> Synthetic)


In [ ]:
def load_dataset():
    # Returns: x_train, y_train, x_test, y_test, dataset_name

    # Option 1: TFDS MNIST
    try:
        import tensorflow_datasets as tfds
        ds_train, ds_test = tfds.load('mnist', split=['train', 'test'], as_supervised=True)

        x_train, y_train = [], []
        for x, y in tfds.as_numpy(ds_train):
            x_train.append(x)
            y_train.append(y)

        x_test, y_test = [], []
        for x, y in tfds.as_numpy(ds_test):
            x_test.append(x)
            y_test.append(y)

        x_train = np.stack(x_train).astype(np.float32)
        y_train = np.array(y_train).astype(np.int32)
        x_test = np.stack(x_test).astype(np.float32)
        y_test = np.array(y_test).astype(np.int32)

        # MNIST: (N,28,28,1), uint8
        return x_train, y_train, x_test, y_test, 'tfds:mnist'
    except Exception as e:
        print('TFDS MNIST unavailable:', type(e).__name__)

    # Option 2: sklearn digits
    try:
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split

        data = load_digits()
        x = data.images.astype(np.float32)  # (N,8,8)
        y = data.target.astype(np.int32)

        # Expand channel dim to NHWC
        x = x[..., None]
        x_train, x_test, y_train, y_test = train_test_split(
            x, y, test_size=0.2, random_state=42, stratify=y
        )
        return x_train, y_train, x_test, y_test, 'sklearn:digits'
    except Exception as e:
        print('sklearn digits unavailable:', type(e).__name__)

    # Option 3: synthetic fallback
    rng = np.random.default_rng(0)
    n_train, n_test, h, w, c = 4096, 1024, 16, 16, 1
    n_classes = 10

    x_train = rng.normal(size=(n_train, h, w, c)).astype(np.float32)
    y_train = rng.integers(0, n_classes, size=(n_train,), endpoint=False).astype(np.int32)
    x_test = rng.normal(size=(n_test, h, w, c)).astype(np.float32)
    y_test = rng.integers(0, n_classes, size=(n_test,), endpoint=False).astype(np.int32)
    return x_train, y_train, x_test, y_test, 'synthetic'

x_train_np, y_train_np, x_test_np, y_test_np, dataset_name = load_dataset()
print('dataset:', dataset_name)
print('train:', x_train_np.shape, y_train_np.shape)
print('test :', x_test_np.shape, y_test_np.shape)


## Section 3: Preprocessing


In [ ]:
def preprocess_images(x_train_np, x_test_np):
    # scale and standardize from train statistics only
    x_train = jnp.array(x_train_np)
    x_test = jnp.array(x_test_np)

    # Handle ranges: MNIST is [0,255], digits is [0,16], synthetic around N(0,1)
    train_max = float(jnp.max(x_train))
    if train_max > 2.0:
        x_train = x_train / train_max
        x_test = x_test / train_max

    mean = jnp.mean(x_train, axis=(0,1,2), keepdims=True)
    std = jnp.std(x_train, axis=(0,1,2), keepdims=True) + 1e-6

    x_train = (x_train - mean) / std
    x_test = (x_test - mean) / std
    return x_train, x_test

x_train, x_test = preprocess_images(x_train_np, x_test_np)
y_train = jnp.array(y_train_np)
y_test = jnp.array(y_test_np)

num_classes = int(jnp.max(y_train)) + 1
print('num_classes:', num_classes)
print('train stats:', float(jnp.mean(x_train)), float(jnp.std(x_train)))


### Exercise 1
Write a function that one-hot encodes labels and verify shape `(N, num_classes)`.


In [ ]:
# Exercise solution

def one_hot(y, num_classes):
    return jax.nn.one_hot(y, num_classes)

y_oh = one_hot(y_train[:8], num_classes)
print(y_oh.shape)
print(y_train[:3], y_oh[:3])


## Section 4: Data Augmentation + Batching


In [ ]:
def random_hflip(key, x, p=0.5):
    n = x.shape[0]
    flips = random.bernoulli(key, p=p, shape=(n,1,1,1))
    return jnp.where(flips, x[:, :, ::-1, :], x)

def random_brightness(key, x, delta=0.1):
    noise = random.uniform(key, (x.shape[0],1,1,1), minval=-delta, maxval=delta)
    return x + noise

@jit
def clip_batch(x):
    return jnp.clip(x, -4.0, 4.0)


def make_batches(key, x, y, batch_size=128, shuffle=True, augment=False):
    n = x.shape[0]
    idx = jnp.arange(n)
    if shuffle:
        idx = random.permutation(key, idx)

    steps = n // batch_size
    for i in range(steps):
        b = idx[i*batch_size:(i+1)*batch_size]
        xb, yb = x[b], y[b]
        if augment:
            key1, key2 = random.split(random.fold_in(key, i))
            xb = random_hflip(key1, xb)
            xb = random_brightness(key2, xb)
            xb = clip_batch(xb)
        yield xb, yb


### Exercise 2
Add a random crop step for images where `H,W >= 12`.


In [ ]:
# Exercise reference implementation

def random_crop_2d(key, x, crop_h, crop_w):
    # x: (B,H,W,C)
    b, h, w, c = x.shape
    if crop_h > h or crop_w > w:
        return x
    k1, k2 = random.split(key)
    top = random.randint(k1, (), 0, h - crop_h + 1)
    left = random.randint(k2, (), 0, w - crop_w + 1)
    return x[:, top:top+crop_h, left:left+crop_w, :]

k = random.PRNGKey(123)
xb, yb = next(make_batches(k, x_train, y_train, batch_size=32, shuffle=False, augment=False))
print('before:', xb.shape)
if xb.shape[1] >= 12 and xb.shape[2] >= 12:
    xb2 = random_crop_2d(k, xb, 12, 12)
    print('after :', xb2.shape)


## Section 5: CNN Classifier in Flax


In [ ]:
class SmallCNN(nn.Module):
    num_classes: int

    @nn.compact
    def __call__(self, x, train=True):
        x = nn.Conv(features=32, kernel_size=(3,3), padding='SAME')(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2,2), strides=(2,2))

        x = nn.Conv(features=64, kernel_size=(3,3), padding='SAME')(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2,2), strides=(2,2))

        x = x.reshape((x.shape[0], -1))
        x = nn.Dense(128)(x)
        x = nn.relu(x)
        logits = nn.Dense(self.num_classes)(x)
        return logits

model = SmallCNN(num_classes=num_classes)
key = random.PRNGKey(0)
params = model.init(key, x_train[:8], True)
print('Initialized parameters')


In [ ]:
optimizer = optax.adamw(1e-3)
opt_state = optimizer.init(params)

@jit
def loss_and_acc(params, x, y):
    logits = model.apply(params, x, False)
    loss = jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, y))
    preds = jnp.argmax(logits, axis=-1)
    acc = jnp.mean((preds == y).astype(jnp.float32))
    return loss, acc

@jit
def train_step(params, opt_state, x, y):
    def loss_fn(p):
        logits = model.apply(p, x, True)
        return jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, y))

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


## Section 6: Train + Evaluate


In [ ]:
num_epochs = 3
batch_size = 128
key = random.PRNGKey(42)

for epoch in range(1, num_epochs + 1):
    key, k = random.split(key)
    train_losses = []

    for xb, yb in make_batches(k, x_train, y_train, batch_size=batch_size, shuffle=True, augment=True):
        params, opt_state, loss = train_step(params, opt_state, xb, yb)
        train_losses.append(loss)

    # Evaluate on test set
    test_losses = []
    test_accs = []
    key, k2 = random.split(key)
    for xb, yb in make_batches(k2, x_test, y_test, batch_size=batch_size, shuffle=False, augment=False):
        l, a = loss_and_acc(params, xb, yb)
        test_losses.append(l)
        test_accs.append(a)

    print(
        f"epoch={epoch} "
        f"train_loss={float(jnp.mean(jnp.stack(train_losses))):.4f} "
        f"test_loss={float(jnp.mean(jnp.stack(test_losses))):.4f} "
        f"test_acc={float(jnp.mean(jnp.stack(test_accs))):.4f}"
    )


## Section 7: Text Dataset Mini-Pipeline (Toy Corpus)

For text, tokenization is typically outside JAX. Here we keep a tiny in-notebook tokenizer to demonstrate shaping/padding/targets.


In [ ]:
corpus = [
    'jax is fast for array programs',
    'flax is a neural network library for jax',
    'optax provides optimizers',
    'transformers need attention masks',
]

# tiny word-level vocab
special = ['<pad>', '<unk>']
words = sorted(set(' '.join(corpus).split()))
vocab = special + words
stoi = {w:i for i,w in enumerate(vocab)}

PAD = stoi['<pad>']
UNK = stoi['<unk>']

def encode(text):
    return [stoi.get(w, UNK) for w in text.split()]

def pad(seq, T, pad_id=PAD):
    seq = seq[:T]
    return seq + [pad_id] * (T - len(seq))

T = 10
toks = jnp.array([pad(encode(s), T) for s in corpus], dtype=jnp.int32)
input_ids = toks[:, :-1]
target_ids = toks[:, 1:]
loss_mask = (target_ids != PAD).astype(jnp.float32)

print('vocab size:', len(vocab))
print('input shape:', input_ids.shape, 'target shape:', target_ids.shape)


### Exercise 3
Create a causal+padding mask from `input_ids` with shape `(B,1,T,T)`.


In [ ]:
# Exercise solution

def causal_padding_mask(tokens, pad_id=0):
    B, T = tokens.shape
    pad_mask = (tokens != pad_id).astype(jnp.float32)[:, None, None, :]  # (B,1,1,T)
    causal = jnp.tril(jnp.ones((T, T), dtype=jnp.float32))[None, None, :, :]  # (1,1,T,T)
    return pad_mask * causal

m = causal_padding_mask(input_ids, PAD)
print(m.shape)


## Section 8: What to Keep in JAX vs Outside JAX

Keep **outside JAX**:
- File IO, dataframe joins, regex-heavy parsing, tokenizer training

Keep **inside JAX**:
- Batch math with stable shapes
- Augmentation ops used per-step
- Mask and target construction tied to model compute

This gives clean, fast, and maintainable pipelines.
